Импорт нужных библиотек:
- Pandas для работы с файлом формата .csv
- Psycopg2 для работы с базой данных
- NumPy для работы с полученными данными

In [1]:
import pandas as pd
import psycopg2
import numpy as np
#pip install psycopg2-binary

Получение данных из файла .csv

In [2]:
df = pd.read_csv('tourism_202603151343.csv', sep=";")

Краткий анализ:
- Количество строк для осознания масштаба действ
- Пример строк для осознания типа данных
- Показ строк, где есть нулевые значения (их нельзя указывать с пометкой "not null")

In [3]:
print("Количество строк - ", len(df))
print("Примеры строк: ")
for i in range(10):
    print(df.iloc[10+i].to_dict())
print("Строки, где есть нулевые значений", df.columns[df.isnull().any()])

Количество строк -  1079177
Примеры строк: 
{'index': 15289, 'territory_code': 22701000, 'territory_name': 'город Нижний Новгород', 'date_of_arrival': '2022-12-31', 'trip_type': 'Поездка внутри страны', 'visit_type': 'Турист', 'home_country': 'Россия', 'home_region': 'Ярославская область', 'home_city': nan, 'goal': 'Посещение друзей и родственников', 'gender': 'мужской', 'tourist_age': 'от 45 до 54 лет', 'income': 'до 30 тыс. руб.', 'days_cnt': 4, 'visitors_cnt': 5, 'spent': 0.0122229}
{'index': 19017, 'territory_code': 22701000, 'territory_name': 'город Нижний Новгород', 'date_of_arrival': '2022-12-31', 'trip_type': 'Поездка внутри страны', 'visit_type': 'Турист', 'home_country': 'Россия', 'home_region': 'Нижегородская область', 'home_city': 'город Дзержинск', 'goal': 'Посещение друзей и родственников', 'gender': 'мужской', 'tourist_age': 'от 18 до 24 лет', 'income': 'свыше 100 тыс. руб.', 'days_cnt': 7, 'visitors_cnt': 4, 'spent': 0.14681829}
{'index': 14342, 'territory_code': 227010

Предположим, что некоторые поля заполнены одним из предопределённых значений:
- territory_code и territory_name
- trip_type
- visit_type
- home_country
- home_region
- home_city
- goal
- gender
- tourist_age
- income

In [4]:
print("territory_code и territory_name: ", df['territory_code'].unique().tolist(),df['territory_name'].unique().tolist())
print("trip_type: ", df['trip_type'].unique().tolist())
print("visit_type: ", df['visit_type'].unique().tolist())
print("home_country: ", df['home_country'].unique().tolist())
print("home_region: ", df['home_region'].unique().tolist())
print("home_city: ", df['home_city'].unique().tolist())
print("goal: ", df['goal'].unique().tolist())
print("gender: ", df['gender'].unique().tolist())
print("tourist_age: ", df['tourist_age'].unique().tolist())
print("income: ", df['income'].unique().tolist())

territory_code и territory_name:  [22701000] ['город Нижний Новгород']
trip_type:  ['Поездка внутри страны', 'Международная поездка']
visit_type:  ['Турист', 'Экскурсант']
home_country:  ['Россия', 'Узбекистан', 'Германия', 'Норвегия', 'Египет', 'Италия', 'Грузия', 'Белоруссия', 'Чехия', 'Таджикистан', 'Турция', 'Болгария', 'Азербайджан', 'Нидерланды', 'США', 'Армения', 'Польша', 'ОАЭ', 'Сербия', 'Кипр', 'Китай\xa0(Китайская Народная Республика)', 'Шри-Ланка', 'Ирак', 'Франция', 'Испания', 'Финляндия', 'Иордания', 'Эстония', 'Израиль', 'Ангола', 'Венгрия', 'Саудовская Аравия', 'Киргизия', 'Дания', 'Индонезия', 'Латвия', 'Катар', 'Швейцария', 'Казахстан', 'Республика Корея', 'Гонконг', 'Черногория', 'Молдавия', 'Таиланд', 'Босния и Герцеговина', 'Великобритания', 'Украина', 'Словения', 'Бельгия', 'Бахрейн', 'Малайзия', 'Индия', 'Албания', 'Япония', 'Фиджи', 'Камерун', 'Танзания', 'Филиппины', 'Литва', 'Кот-д’Ивуар', 'Португалия', 'Гватемала', 'Туркмения', 'Чили', 'Люксембург', 'Уганда',

У столбцов territory_code и territory_name только одно значение - 22701000 и 'город Нижний Новгород' соответственно

У столбца trip_type два значения:
- 'Поездка внутри страны'
- 'Международная поездка'

У столбца visit_type два значения:
- 'Турист'
- 'Экскурсант'

У столбца home_country 106 значений

У столбца home_region 85 значений - 83 региона России (это значит, что регион туриста не указывается, если он не из России) и подставные значения nan и 0

У столбца home_city 53 значения - 51 район Нижегородской области (это значит, что город туриста не указывается, если он не из Нижегородской области) и подставные значения nan и 0

У столбца trip_type три значения:
- 'Посещение друзей и родственников'
- 'Туризм'
- 'Командировка'

У столбца gender четыре значения:
- 'женский'
- 'мужской'

и подставные значения nan и 0

У столбца tourist_age 9 значений - семь возрастных промежутков:
- 'от 25 до 34 лет'
- 'от 35 до 44 лет'
- 'от 55 до 63 лет'
- 'свыше 64 лет'
- 'от 45 до 54 лет'
- 'от 18 до 24 лет'
- 'до 18 лет'

и подставные значения nan и 0

У столбца income 7 значений - пять зарплатных промежутков:
- 'до 30 тыс. руб.'
- 'от 30 до 50 тыс. руб.'
- 'от 70 до 100 тыс. руб.'
- 'свыше 100 тыс. руб.'
- 'от 50 до 70 тыс. руб.'

и подставные значения nan и 0

Этих данных достаточно для создания схемы базы данных:

![схема базы данных](DBplan.png)

Теперь можно создать базу данных:



In [ ]:
"""
CREATE TABLE trip_type (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL UNIQUE
);

CREATE TABLE visit_type (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL UNIQUE
);

CREATE TABLE country (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL UNIQUE
);

CREATE TABLE region (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL UNIQUE
);

CREATE TABLE city (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL UNIQUE
);

CREATE TABLE goal (
    id SERIAL PRIMARY KEY,
    name VARCHAR(200) NOT NULL UNIQUE
);

CREATE TABLE gender (
    id SERIAL PRIMARY KEY,
    name VARCHAR(20) NOT NULL UNIQUE
);

CREATE TABLE tourist_age (
    id SERIAL PRIMARY KEY,
    name VARCHAR(50) NOT NULL UNIQUE
);

CREATE TABLE income (
    id SERIAL PRIMARY KEY,
    name VARCHAR(50) NOT NULL UNIQUE
);

CREATE TABLE tourists (
    index SERIAL PRIMARY KEY,

    territory_code VARCHAR(50),
    territory_name VARCHAR(200),
    date_of_arrival DATE,

    trip_type_id INTEGER REFERENCES trip_type(id),
    visit_type_id INTEGER REFERENCES visit_type(id),
    home_country_id INTEGER REFERENCES country(id),
    home_region_id INTEGER REFERENCES region(id),
    home_city_id INTEGER REFERENCES city(id),
    goal_id INTEGER REFERENCES goal(id),
    gender_id INTEGER REFERENCES gender(id),
    tourist_age_id INTEGER REFERENCES tourist_age(id),
    income_id INTEGER REFERENCES income(id),

    days_cnt INTEGER,
    visitors_cnt INTEGER,
    spent DECIMAL(10, 8),
);
"""

![Созданная база данных](CreatedDB.png)

Для того, чтобы добавить данные в базу данных, все nan и 0 нужно заменить на "Не указан"

Строки, где есть нулевые значений Index(['home_region', 'home_city', 'gender', 'tourist_age', 'income', 'spent'], dtype='str'):

In [5]:
df['home_region'] = df['home_region'].replace({
    np.nan: 'Не указан',
    '0': 'Не указан'
})
print("Есть ли NaN или 0 в home_region? ", np.nan in df['home_region'].unique().tolist() or '0' in df['home_region'].unique().tolist())

df['home_city'] = df['home_city'].replace({
    np.nan: 'Не указан',
    '0': 'Не указан'
})
print("Есть ли NaN или 0 в home_city? ", np.nan in df['home_city'].unique().tolist() or '0' in df['home_city'].unique().tolist())

df['gender'] = df['gender'].replace({
    np.nan: 'Не указан',
    '0': 'Не указан'
})
print("Есть ли NaN или 0 в gender? ", np.nan in df['gender'].unique().tolist() or '0' in df['gender'].unique().tolist())

df['tourist_age'] = df['tourist_age'].replace({
    np.nan: 'Не указан',
    '0': 'Не указан'
})
print("Есть ли NaN или 0 в tourist_age? ", np.nan in df['tourist_age'].unique().tolist() or '0' in df['tourist_age'].unique().tolist())

df['income'] = df['income'].replace({
    np.nan: 'Не указан',
    '0': 'Не указан'
})
print("Есть ли NaN или 0 в income? ", np.nan in df['income'].unique().tolist() or '0' in df['income'].unique().tolist())
#В столбце spent допустимо отсутствие значения

print("Строки, где есть нулевые значений", df.columns[df.isnull().any()])

Есть ли NaN или 0 в home_region?  False
Есть ли NaN или 0 в home_city?  False
Есть ли NaN или 0 в gender?  False
Есть ли NaN или 0 в tourist_age?  False
Есть ли NaN или 0 в income?  False
Строки, где есть нулевые значений Index(['spent'], dtype='str')


Теперь можно заполнять базу данных

In [9]:
#Подключение

connection = psycopg2.connect(
    host="rc1b-4r4ggpum17pegln9.mdb.yandexcloud.net",
    database="tourists",
    user="andrzej",
    password="postgres",
    port="6432",
)

cursor = connection.cursor()

#Добавление данных в посредственные таблицы
trip_type_query = "INSERT INTO tourists.trip_type (name) VALUES "
for i in df['trip_type'].unique().tolist():
    trip_type_query += f"('{i}'), ";
trip_type_query = trip_type_query[:-2]+';'
cursor.execute(trip_type_query)

visit_type = "INSERT INTO tourists.visit_type (name) VALUES "
for i in df['visit_type'].unique().tolist():
    visit_type += f"('{i}'), ";
visit_type= visit_type[:-2]+';'
cursor.execute(visit_type)

county_query = "INSERT INTO tourists.country (name) VALUES "
for i in df['home_country'].unique().tolist():
    county_query += f"('{i}'), ";
county_query= county_query[:-2]+';'
cursor.execute(county_query)

region_query = "INSERT INTO tourists.region (name) VALUES "
for i in df['home_region'].unique().tolist():
    region_query += f"('{i}'), ";
region_query= region_query[:-2]+';'
cursor.execute(region_query)

city_query = "INSERT INTO tourists.city (name) VALUES "
for i in df['home_city'].unique().tolist():
    city_query += f"('{i}'), ";
city_query= city_query[:-2]+';'
cursor.execute(city_query)

goal_query = "INSERT INTO tourists.goal (name) VALUES "
for i in df['goal'].unique().tolist():
    goal_query += f"('{i}'), ";
goal_query= goal_query[:-2]+';'
cursor.execute(goal_query)

gender_query = "INSERT INTO tourists.gender (name) VALUES "
for i in df['gender'].unique().tolist():
    gender_query += f"('{i}'), ";
gender_query= gender_query[:-2]+';'
cursor.execute(gender_query)

tourist_age_query = "INSERT INTO tourists.tourist_age (name) VALUES "
for i in df['tourist_age'].unique().tolist():
    tourist_age_query += f"('{i}'), ";
tourist_age_query= tourist_age_query[:-2]+';'
cursor.execute(tourist_age_query)

income_query = "INSERT INTO tourists.income (name) VALUES "
for i in df['income'].unique().tolist():
    income_query += f"('{i}'), ";
income_query= income_query[:-2]+';'
cursor.execute(income_query)

# cursor.execute("SELECT version();")
# db_version = cursor.fetchone()
# print(f"Connected to PostgreSQL. Server version: {db_version[0]}")

connection.commit()
cursor.close()
connection.close()
print("Database connection closed safely.")

Database connection closed safely.


In [10]:
"""connection = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="postgres",
    port="5432",
)"""
connection = psycopg2.connect(
    host="rc1b-4r4ggpum17pegln9.mdb.yandexcloud.net",
    database="tourists",
    user="andrzej",
    password="postgres",
    port="6432",
)

cursor = connection.cursor()

values_list = []
i = 1
for _, row in df.iterrows():
    i+=1
    values_list.append(f"""(
        '{row['territory_code']}', '{row['territory_name']}', '{row['date_of_arrival']}',
        (SELECT id FROM tourists.trip_type WHERE name = '{row['trip_type']}'),
        (SELECT id FROM tourists.visit_type WHERE name = '{row['visit_type']}'),
        (SELECT id FROM tourists.country WHERE name = '{row['home_country']}'),
        (SELECT id FROM tourists.region WHERE name = '{row['home_region']}'),
        (SELECT id FROM tourists.city WHERE name = '{row['home_city']}'),
        (SELECT id FROM tourists.goal WHERE name = '{row['goal']}'),
        (SELECT id FROM tourists.gender WHERE name = '{row['gender']}'),
        (SELECT id FROM tourists.tourist_age WHERE name = '{row['tourist_age']}'),
        (SELECT id FROM tourists.income WHERE name = '{row['income']}'),
        {row['days_cnt']}, {row['visitors_cnt']}, {'NULL' if pd.isna(row['spent']) else row['spent']}
    )""")
    if i%1000==0:
        query = f"""
            INSERT INTO tourists.tourists (
                territory_code, territory_name, date_of_arrival,
                trip_type_id, visit_type_id, home_country_id, home_region_id, home_city_id,
                goal_id, gender_id, tourist_age_id, income_id,
                days_cnt, visitors_cnt, spent
            ) VALUES {', '.join(values_list)};
        """
        cursor.execute(query)
        connection.commit()
        values_list = []
        print(i, end=" ")
query = f"""
    INSERT INTO tourists.tourists (
        territory_code, territory_name, date_of_arrival,
        trip_type_id, visit_type_id, home_country_id, home_region_id, home_city_id,
        goal_id, gender_id, tourist_age_id, income_id,
        days_cnt, visitors_cnt, spent
    ) VALUES {', '.join(values_list)};
"""
cursor.execute(query)
connection.commit()

cursor.close()
connection.close()
print("Данные перемещены")

1000 2000 3000 4000 5000 6000 7000 8000 9000 10000 11000 12000 13000 14000 15000 16000 17000 18000 19000 20000 21000 22000 23000 24000 25000 26000 27000 28000 29000 30000 31000 32000 33000 34000 35000 36000 37000 38000 39000 40000 41000 42000 43000 44000 45000 46000 47000 48000 49000 50000 51000 52000 53000 54000 55000 56000 57000 58000 59000 60000 61000 62000 63000 64000 65000 66000 67000 68000 69000 70000 71000 72000 73000 74000 75000 76000 77000 78000 79000 80000 81000 82000 83000 84000 85000 86000 87000 88000 89000 90000 91000 92000 93000 94000 95000 96000 97000 98000 99000 100000 101000 102000 103000 104000 105000 106000 107000 108000 109000 110000 111000 112000 113000 114000 115000 116000 117000 118000 119000 120000 121000 122000 123000 124000 125000 126000 127000 128000 129000 130000 131000 132000 133000 134000 135000 136000 137000 138000 139000 140000 141000 142000 143000 144000 145000 146000 147000 148000 149000 150000 151000 152000 153000 154000 155000 156000 157000 158000 15